In [16]:
# Import libraries
import pandas as pd
import json
import requests
import os
from pathlib import Path
import logging
from datetime import datetime

In [2]:
# Logging to track ETL process
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Create directory structure
DATA_DIR = Path("ayurvedic_data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "output"

for dir_path in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    logger.info(f"Created directory: {dir_path}")

logger.info("Setup complete! Ready to extract data.")

2026-08-21 10:45:31,867 - INFO - Created directory: ayurvedic_data\raw
2026-08-21 10:45:31,871 - INFO - Created directory: ayurvedic_data\processed
2026-08-21 10:45:31,873 - INFO - Created directory: ayurvedic_data\output
2026-08-21 10:45:31,875 - INFO - Setup complete! Ready to extract data.


In [3]:
# Extract from the Amidha Ayurveda database
def extract_from_json_url(url, save_raw=True):
    """Extract data from a JSON API endpoint"""

    logger.info(f"Extracting data from: {url}")

    try:
        response = requests.get(url)
        response.raise_for_status()   # raise an exception
        data = response.json()
        logger.info(f"Successfully extracted {len(data)} records")

        # Save raw data
        if save_raw:
            raw_file = RAW_DIR / "ayurveda_herbs_raw.json"
            with open(raw_file, 'w') as f:
                json.dump(data, f, indent=2)
            logger.info(f"Raw data saved to: {raw_file}")
        return data
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to extract data: {e}")
        return None

# Extract data
url = "https://raw.githubusercontent.com/sciencewithsaucee-sudo/herb-database/main/herb.json"
raw_herb_data = extract_from_json_url(url)

# Inspect data
if raw_herb_data:
    print(f"Number of herbs: {len(raw_herb_data)}")
    print("\nFirst herb sample:")
    print(json.dumps(raw_herb_data[0], indent=2)[:1000])


2026-08-21 11:14:17,084 - INFO - Extracting data from: https://raw.githubusercontent.com/sciencewithsaucee-sudo/herb-database/main/herb.json
2026-08-21 11:14:17,896 - INFO - Successfully extracted 360 records
2026-08-21 11:14:17,941 - INFO - Raw data saved to: ayurvedic_data\raw\ayurveda_herbs_raw.json


Number of herbs: 360

First herb sample:
{
  "name": "Tulsi",
  "botanical_name": "Ocimum sanctum",
  "family": "Lamiaceae",
  "english_name": "Holy Basil",
  "sanskrit_synonyms": [
    "Surasa",
    "Apetarakshasi",
    "Bhuteshta",
    "Gauri"
  ],
  "part_used": [
    "Patra",
    "Pushpa",
    "Moola",
    "Bija"
  ],
  "main_indications": [
    "Kasa",
    "Shwasa",
    "Jwara",
    "Hikka"
  ],
  "image": "",
  "link": "https://www.amidhaayurveda.com/p/tulsi.html",
  "preview": "Tulsi (Ocimum sanctum) is known as the 'Queen of Herbs' in Ayurveda. It boosts immunity, combats respiratory disorders, and has adaptogenic properties.",
  "pacify": [
    "Kapha",
    "Vata"
  ],
  "aggravate": [
    "Pitta"
  ],
  "tridosha": false,
  "rasa": [
    "Katu",
    "Tikta"
  ],
  "guna": [
    "Laghu",
    "Ruksha"
  ],
  "virya": "Ushna",
  "vipaka": "Katu",
  "prabhav": [
    "rasayan",
    "Krimighna",
    "Kasahara",
    "Shoolaghna"
  ]
}


In [4]:
# Multi-source extraction
def extract_multiple_sources():
    """Extract data from multiple"""

    all_herbs = []

    # Primary JSON dataset
    primary_data = extract_from_json_url(url)
    if primary_data:
        for herb in primary_data:
            herb['_source'] = 'amidha_ayurveda'
        all_herbs.extend(primary_data)

    # Extract from a CSV file
    # csv_file = RAW_DIR / "additional_herbs.csv"
    # if csv_file.exists():
    #     additional_data = pd.read_csv(csv_file)
    #     all_herbs.extend(additional_data.to_dict('records'))
    
    # Query an API
    # api_data = query_herb_api()
    # all_herbs.extend(api_data)

    logger.info(f"Total records from all sources: {len(all_herbs)}")
    return all_herbs

In [5]:
# Load data for transformation
def load_to_dataframe(data):
    """Convert JSON data to a DataFrame"""
    if not data:
        logger.error("No data to load")
        return None

    df = pd.DataFrame(data)
    logger.info(f"Loaded {len(df)} records in DataFrame")
    logger.info(f"Columns: {df.columns.tolist()}")

    return df

herb_df = load_to_dataframe(raw_herb_data)
herb_df.head()

2026-08-21 11:54:31,865 - INFO - Loaded 360 records in DataFrame
2026-08-21 11:54:31,871 - INFO - Columns: ['name', 'botanical_name', 'family', 'english_name', 'sanskrit_synonyms', 'part_used', 'main_indications', 'image', 'link', 'preview', 'pacify', 'aggravate', 'tridosha', 'rasa', 'guna', 'virya', 'vipaka', 'prabhav']


,name,botanical_name,family,english_name,sanskrit_synonyms,part_used,main_indications,image,link,preview,pacify,aggravate,tridosha,rasa,guna,virya,vipaka,prabhav
0,Tulsi,Ocimum sanctum,Lamiaceae,Holy Basil,"[Surasa, Apetarakshasi, Bhuteshta, Gauri]","[Patra, Pushpa, Moola, Bija]","[Kasa, Shwasa, Jwara, Hikka]",,https://www.amidhaayurveda.com/p/tulsi.html,Tulsi (Ocimum sanctum) is known as the 'Queen ...,"[Kapha, Vata]",[Pitta],False,"[Katu, Tikta]","[Laghu, Ruksha]",Ushna,Katu,"[rasayan, Krimighna, Kasahara, Shoolaghna]"
1,Amla,Emblica officinalis,Euphorbiaceae,Indian Gooseberry,"[Amalaki, Dhatri, Vayastha, Vrishya]",[Phala],"[Raktapitta, Amlapitta, Prameha, Trishna]",,https://www.amidhaayurveda.com/p/amla.html,Amla (Emblica officinalis) is a potent source ...,[tridosha],[],True,[Amla],"[Laghu, Snigdha]",Sheeta,Madhura,"[rasayan, Chakshushya, Hridaya, Deepana]"
2,Ashwagandha,Withania somnifera,Solanaceae,Indian Ginseng,"[Turagi, Vajigandha, Balada, Kushtagandhini]",[Moola],"[Kshaya, Daurbalya, Vata Vyadhi, Klaibya]",,https://www.amidhaayurveda.com/p/ashwagandha.html,Ashwagandha (Withania somnifera) is a powerful...,"[Vata, Kapha]",[Pitta],False,"[Tikta, Kashaya]","[Guru, Snigdha]",Ushna,Madhura,"[Balya, Medhya, Vrishya, rasayan]"
3,Giloy,Tinospora cordifolia,Menispermaceae,Heart-leaved Moonseed,"[Guduchi, Amrita, Madhuparni, Chinnaruha]",[Kanda],"[Jwara, Kamala, Kushta, Prameha, Vatarakta]",,https://www.amidhaayurveda.com/p/giloy.html,Giloy (Tinospora cordifolia) is a powerful imm...,[tridosha],[],True,[Tikta],"[Laghu, Sara]",Ushna,Madhura,"[Jwaraghna, Raktashodhak, Shothahara, rasayan]"
4,Shatavari,Asparagus racemosus,Asparagaceae,Wild Asparagus,"[Shatamuli, Atirasa, Narayani, Bahusuta]",[Moola],"[Stanyakshaya, Raktapitta, Amlapitta, Shukraks...",,https://www.amidhaayurveda.com/p/shatavari.html,Shatavari (Asparagus racemosus) supports femal...,"[Vata, Pitta]",[Kapha],False,[Madhura],"[Guru, Snigdha]",Sheeta,Madhura,"[Stanyashodhana, Shukrala, rasayan, Varnya]"


In [7]:
# Data quality assessment
def assess_data_quality(df):
    """Perform data quality checks"""
    logger.info("=== DATA QUALITY ASSESSMENT ===")

    # Check for missing values
    missing_counts = df.isnull().sum()
    missing_pct = (missing_counts / len(df)) *  100
    logger.info("\nMissing values:")
    for col in missing_counts.index:
        if missing_counts[col] > 0:
            logger.info(f"   {col}: {missing_counts[0]} ({missing_pct[col]:.1f}%)")

    # Check for duplicates
    duplicates = df.duplicated(subset=['name']).sum()
    logger.info(f"\nDuplicate herb names: {duplicates}")

    # Check data types
    logger.info(f"\nData types:\n{df.dtypes}")

    # Check for cols
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, list)).any():
            logger.info(f"   {col}: contains list values")

    return {
        'missing_values': missing_counts,
        'duplicates': duplicates,
        'dtypes': df.dtypes
    }

quality_report = assess_data_quality(herb_df)
            
    

2026-08-21 12:11:49,120 - INFO - === DATA QUALITY ASSESSMENT ===
2026-08-21 12:11:49,124 - INFO - 
Missing values:
2026-08-21 12:11:49,136 - INFO - 
Duplicate herb names: 0
2026-08-21 12:11:49,141 - INFO - 
Data types:
name                    str
botanical_name          str
family                  str
english_name            str
sanskrit_synonyms    object
part_used            object
main_indications     object
image                   str
link                    str
preview                 str
pacify               object
aggravate            object
tridosha               bool
rasa                 object
guna                 object
virya                   str
vipaka                  str
prabhav              object
dtype: object
2026-08-21 12:11:49,148 - INFO -    sanskrit_synonyms: contains list values
2026-08-21 12:11:49,150 - INFO -    part_used: contains list values
2026-08-21 12:11:49,152 - INFO -    main_indications: contains list values
2026-08-21 12:11:49,156 - INFO -    pacify: 

In [8]:
# Transformation functions
def transform_herb_data(df):
    """Transform herb data"""

    logger.info("Starting transformation pipeline...")
    transformed_df = df.copy()

    # Handle lists
    list_columns = ['sanskrit_synonyms', 'part_used', 'main_indications', 'rasa', 
                    'guna', 'pacify', 'aggravate', 'prabhav']
    for col in list_columns:
        if col in transformed_df.columns:
            # Convert list to comma-seperated string
            transformed_df[f'{col}_as_text'] = transformed_df[col].apply(
                lambda x: ', '.join(x) if isinstance(x, list) else x)
            logger.info(f"   Converted {col} to text format")

    # Extract key properties
    property_mapping = {
        'virya': 'virya',
        'vipaka': 'vipaka',
        'tridosha': 'tridosha'
    }

    for source_col, target_col in property_mapping.items():
        if source_col in transformed_df.columns:
            transformed_df[target_col] = transformed_df[source_col]
            logger.info(f"   Mapped {source_col} to {target_col}")

    # Strip whitespace
    text_columns = ['name', 'botanical_name', 'english_name', 'family']
    for col in text_columns:
        if col in transformed_df.columns:
            transformed_df[col] = transformed_df[col].str.strip()
            logger.info(f"   Standardised {col}")

    # Create therapeutic_uses col
    if 'main_indications' in transformed_df.columns:
        transformed_df['therapeutic_uses'] = transformed_df['main_indications'].apply(
            lambda x: ', '.join(x) if isinstance(x, list) else x
        )
        logger.info("   Created therapeutic_uses from main_indications")

    # Handle missing values
    categorical_cols = ['virya', 'vipaka']
    for col in categorical_cols:
        if col in transformed_df.columns:
            transformed_df[col] = transformed_df[col].fillna('Not specified')

    # Add metadata
    transformed_df['_extract_date'] = datetime.now().strftime('%Y-%m-%d')
    transformed_df['_source'] = 'amidha_ayurveda_v2.0'

    logger.info("Transformation complete!")
    return transformed_df

# Apply transformations
transformed_herb_df = transform_herb_data(herb_df)
transformed_herb_df.head()

2026-08-21 14:17:12,598 - INFO - Starting transformation pipeline...
2026-08-21 14:17:12,612 - INFO -    Converted sanskrit_synonyms to text format
2026-08-21 14:17:12,618 - INFO -    Converted part_used to text format
2026-08-21 14:17:12,621 - INFO -    Converted main_indications to text format
2026-08-21 14:17:12,625 - INFO -    Converted rasa to text format
2026-08-21 14:17:12,628 - INFO -    Converted guna to text format
2026-08-21 14:17:12,631 - INFO -    Converted pacify to text format
2026-08-21 14:17:12,633 - INFO -    Converted aggravate to text format
2026-08-21 14:17:12,635 - INFO -    Converted prabhav to text format
2026-08-21 14:17:12,640 - INFO -    Mapped virya to virya
2026-08-21 14:17:12,642 - INFO -    Mapped vipaka to vipaka
2026-08-21 14:17:12,643 - INFO -    Mapped tridosha to tridosha
2026-08-21 14:17:12,646 - INFO -    Standardised name
2026-08-21 14:17:12,649 - INFO -    Standardised botanical_name
2026-08-21 14:17:12,651 - INFO -    Standardised english_name
2

,name,botanical_name,family,english_name,sanskrit_synonyms,part_used,main_indications,image,link,preview,...,part_used_as_text,main_indications_as_text,rasa_as_text,guna_as_text,pacify_as_text,aggravate_as_text,prabhav_as_text,therapeutic_uses,_extract_date,_source
0,Tulsi,Ocimum sanctum,Lamiaceae,Holy Basil,"[Surasa, Apetarakshasi, Bhuteshta, Gauri]","[Patra, Pushpa, Moola, Bija]","[Kasa, Shwasa, Jwara, Hikka]",,https://www.amidhaayurveda.com/p/tulsi.html,Tulsi (Ocimum sanctum) is known as the 'Queen ...,...,"Patra, Pushpa, Moola, Bija","Kasa, Shwasa, Jwara, Hikka","Katu, Tikta","Laghu, Ruksha","Kapha, Vata",Pitta,"rasayan, Krimighna, Kasahara, Shoolaghna","Kasa, Shwasa, Jwara, Hikka",2026-08-21,amidha_ayurveda_v2.0
1,Amla,Emblica officinalis,Euphorbiaceae,Indian Gooseberry,"[Amalaki, Dhatri, Vayastha, Vrishya]",[Phala],"[Raktapitta, Amlapitta, Prameha, Trishna]",,https://www.amidhaayurveda.com/p/amla.html,Amla (Emblica officinalis) is a potent source ...,...,Phala,"Raktapitta, Amlapitta, Prameha, Trishna",Amla,"Laghu, Snigdha",tridosha,,"rasayan, Chakshushya, Hridaya, Deepana","Raktapitta, Amlapitta, Prameha, Trishna",2026-08-21,amidha_ayurveda_v2.0
2,Ashwagandha,Withania somnifera,Solanaceae,Indian Ginseng,"[Turagi, Vajigandha, Balada, Kushtagandhini]",[Moola],"[Kshaya, Daurbalya, Vata Vyadhi, Klaibya]",,https://www.amidhaayurveda.com/p/ashwagandha.html,Ashwagandha (Withania somnifera) is a powerful...,...,Moola,"Kshaya, Daurbalya, Vata Vyadhi, Klaibya","Tikta, Kashaya","Guru, Snigdha","Vata, Kapha",Pitta,"Balya, Medhya, Vrishya, rasayan","Kshaya, Daurbalya, Vata Vyadhi, Klaibya",2026-08-21,amidha_ayurveda_v2.0
3,Giloy,Tinospora cordifolia,Menispermaceae,Heart-leaved Moonseed,"[Guduchi, Amrita, Madhuparni, Chinnaruha]",[Kanda],"[Jwara, Kamala, Kushta, Prameha, Vatarakta]",,https://www.amidhaayurveda.com/p/giloy.html,Giloy (Tinospora cordifolia) is a powerful imm...,...,Kanda,"Jwara, Kamala, Kushta, Prameha, Vatarakta",Tikta,"Laghu, Sara",tridosha,,"Jwaraghna, Raktashodhak, Shothahara, rasayan","Jwara, Kamala, Kushta, Prameha, Vatarakta",2026-08-21,amidha_ayurveda_v2.0
4,Shatavari,Asparagus racemosus,Asparagaceae,Wild Asparagus,"[Shatamuli, Atirasa, Narayani, Bahusuta]",[Moola],"[Stanyakshaya, Raktapitta, Amlapitta, Shukraks...",,https://www.amidhaayurveda.com/p/shatavari.html,Shatavari (Asparagus racemosus) supports femal...,...,Moola,"Stanyakshaya, Raktapitta, Amlapitta, Shukrakshaya",Madhura,"Guru, Snigdha","Vata, Pitta",Kapha,"Stanyashodhana, Shukrala, rasayan, Varnya","Stanyakshaya, Raktapitta, Amlapitta, Shukrakshaya",2026-08-21,amidha_ayurveda_v2.0


In [11]:
# Feature engineering
def add_analytical_features(df):
    """Create features for analysis"""

    logger.info("Adding analytical features...")
    enhanced_df = df.copy()

    # Find herb's importance
    if 'sanskrit_synonyms' in enhanced_df.columns:
        enhanced_df['synonym_count'] = enhanced_df['sanskrit_synonyms'].apply(
            lambda x: len(x) if isinstance(x, list) and x else 0
        )
        logger.info(" Added synonym_count")

    # Count main indications
    if 'main_indications' in enhanced_df.columns:
        enhanced_df['indication_count'] = enhanced_df['main_indications'].apply(
            lambda x: len(x) if isinstance(x, list) and x else 0
        )
        logger.info(" Added indication_count")

    # Herbs specific use (prabhav)
    if 'prabhav' in enhanced_df.columns:
        enhanced_df['has_prabhav'] = enhanced_df['prabhav'].apply(
            lambda x: len(x) > 0 if isinstance(x, list) and x else False
        )
        logger.info(" Added has_prabhav flag")

    # Combine dosha effects in description
    if 'pacify' in enhanced_df.columns and 'aggravate' in enhanced_df.columns:
        def create_dosha_description(row):
            pacify = ', '.join(row['pacify']) if isinstance(row['pacify'], list) else ''
            aggravate = ', '.join(row['aggravate']) if isinstance(row['aggravate'], list) else ''
            if pacify and aggravate:
                return f"Pacifies {pacify}; Aggravates {aggravate}"
            elif pacify:
                return f"Pacifies {pacify}"
            elif aggravate:
                return f"Aggravates {aggravate}"
            return "Not specified"

        enhanced_df['dosha_effects'] = enhanced_df.apply(create_dosha_description, axis=1)
        logger.info(" Added dosha_effects description")

    # Create a 'use_category'
    if 'therapeutic_uses' in enhanced_df.columns:
        immune_keywords = ['kasa', 'shwasa', 'jwara', 'immun']
        respiratory_keywords = ['kasa', 'shwasa', 'hikka']

        def categorise_use(uses_text):
            if not isinstance(uses_text, str):
                return 'Unknown'
            uses_lower = uses_text.lower()
            if any(keyword in uses_lower for keyword in immune_keywords):
                return 'Immune/Respiratory'
            elif any(keyword in uses_lower for keyword in respiratory_keywords):
                return 'Respiratory'
            else:
                return 'Other'

        enhanced_df['use_category'] = enhanced_df['therapeutic_uses'].apply(categorise_use)
        logger.info(" Added use_category")

    return enhanced_df

final_herb_df = add_analytical_features(transformed_herb_df)
logger.info(f"Final dataset shape: {final_herb_df.shape}")
final_herb_df.info()

2026-08-21 14:45:37,790 - INFO - Adding analytical features...
2026-08-21 14:45:37,793 - INFO -    Added synonym_count
2026-08-21 14:45:37,795 - INFO -    Added indication_count
2026-08-21 14:45:37,798 - INFO -    Added has_prabhav flag
2026-08-21 14:45:37,813 - INFO -    Added dosha_effects description
2026-08-21 14:45:37,819 - INFO -   Added use_category
2026-08-21 14:45:37,821 - INFO - Final dataset shape: (360, 34)


<class 'pandas.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   name                       360 non-null    str   
 1   botanical_name             360 non-null    str   
 2   family                     360 non-null    str   
 3   english_name               360 non-null    str   
 4   sanskrit_synonyms          360 non-null    object
 5   part_used                  360 non-null    object
 6   main_indications           360 non-null    object
 7   image                      360 non-null    str   
 8   link                       360 non-null    str   
 9   preview                    360 non-null    str   
 10  pacify                     360 non-null    object
 11  aggravate                  360 non-null    object
 12  tridosha                   360 non-null    bool  
 13  rasa                       360 non-null    object
 14  guna                 

In [19]:
# Load data
def load_transformed_data(df, output_name="ayurvedic_herbs"):
    """Load transformed data"""

    # Save as CSV 
    csv_path = OUTPUT_DIR / f"{output_name}.csv"
    df.to_csv(csv_path, index=False)
    logger.info(f"Saved CSV to: {csv_path}")

    # Save as JSON
    json_path = OUTPUT_DIR / f"{output_name}.json"
    df.to_json(json_path, orient='records', indent=2)
    logger.info(f"Saved JSON to: {json_path}")   

    return {
        'csv': csv_path,
        'json': json_path        
    }

# Load the data
output_files = load_transformed_data(final_herb_df)
print("Data loaded to:")
for fmt, path in output_files.items():
    print(f" {fmt}: {path}")
        

2026-08-21 15:29:13,783 - INFO - Saved CSV to: ayurvedic_data\output\ayurvedic_herbs.csv
2026-08-21 15:29:13,794 - INFO - Saved JSON to: ayurvedic_data\output\ayurvedic_herbs.json


Data loaded to:
 csv: ayurvedic_data\output\ayurvedic_herbs.csv
 json: ayurvedic_data\output\ayurvedic_herbs.json


In [20]:
# Validate data
def validate_dataset(df, original_df):
    """Validate the dataset"""

    logger.info("Running validation checks...")

    # Check row count
    assert len(df) >= len(original_df) * 0.9, "Too many rows lost in transformation"
    logger.info(f" Row count: {len(df)} (from {len(original_df)})")

    # Count columns
    logger.info(f"Columns: {len(df.columns)} (from {len(original_df.columns)})")
    logger.info(f" New columns: {set(df.columns) - set(original_df.columns)}")

    # Check for missing values in critical columns
    critical_columns = ['name', 'botanical_name']
    for col in critical_columns:
        missing = df[col].isnull().sum()
        assert missing == 0, f"Missing values in {col}: {missing}"
        logger.info(f" {col}: not missing values")

    # Check data types
    logger.info(f" Data types validated")

    # Generate summary statistics
    logger.info("\n=== DATASET SUMMARY ===")
    logger.info(f"Total herbs: {len(df)}")
    logger.info(f"Unique families: {df['family'].nunique() if 'family' in df else 'Not available'}")
    logger.info(f"Date range: {df['_extract_date'].iloc[0] if '_extract_date' in df else 'Not available'}")

    # Count by Virya (potency)
    if 'virya' in df:
        logger.info(f"\nVirya (Potency) distribution:")
        virya_counts = df['virya'].value_counts()
        for potency, count in virya_counts.items():
            logger.info(f"  {potency}: {count} ({count/len(df)*100:.1f}%)")
    
    # Count by tridosha classification
    if 'tridosha' in df:
        tridosha_counts = df['tridosha'].value_counts()
        logger.info(f"\nTridosha classification:")
        for classification, count in tridosha_counts.items():
            logger.info(f"  {classification}: {count} ({count/len(df)*100:.1f}%)")
    
    logger.info("\n✓ Validation complete - dataset is ready for use!")
    return True

validate_dataset(final_herb_df, herb_df)
    

2026-08-21 17:15:29,321 - INFO - Running validation checks...
2026-08-21 17:15:29,324 - INFO -  Row count: 360 (from 360)
2026-08-21 17:15:29,326 - INFO - Columns: 34 (from 18)
2026-08-21 17:15:29,330 - INFO -  New columns: {'pacify_as_text', 'main_indications_as_text', 'dosha_effects', 'synonym_count', 'therapeutic_uses', 'prabhav_as_text', 'indication_count', '_source', 'aggravate_as_text', 'use_category', 'has_prabhav', '_extract_date', 'rasa_as_text', 'sanskrit_synonyms_as_text', 'guna_as_text', 'part_used_as_text'}
2026-08-21 17:15:29,349 - INFO -  name: not missing values
2026-08-21 17:15:29,352 - INFO -  botanical_name: not missing values
2026-08-21 17:15:29,353 - INFO -  Data types validated
2026-08-21 17:15:29,354 - INFO - 
=== DATASET SUMMARY ===
2026-08-21 17:15:29,355 - INFO - Total herbs: 360
2026-08-21 17:15:29,362 - INFO - Unique families: 108
2026-08-21 17:15:29,364 - INFO - Date range: 2026-08-21
2026-08-21 17:15:29,366 - INFO - 
Virya (Potency) distribution:
2026-08-2

True

In [22]:
# Analysis on the dataset
def analyse_herb_properties(df):
    """
    Perform a simple analysis to demonstrate the dataset's value.
    """
    logger.info("\n=== ANALYSIS: Top 10 Herbs by Sanskrit Synonyms ===\n")
    
    if 'synonym_count' in df and 'name' in df:
        top_synonyms = df.nlargest(10, 'synonym_count')[['name', 'sanskrit_synonyms_as_text', 'synonym_count']]
        print(top_synonyms.to_string(index=False))
    
    logger.info("\n=== ANALYSIS: Most Common Therapeutic Indications ===\n")
    if 'therapeutic_uses' in df:
        # Split indications and count occurrences
        all_indications = []
        for indications in df['therapeutic_uses'].dropna():
            if isinstance(indications, str):
                all_indications.extend([ind.strip() for ind in indications.split(',')])
        
        from collections import Counter
        indication_counter = Counter(all_indications)
        
        print("Top 10 most common indications:")
        for indication, count in indication_counter.most_common(10):
            print(f"  {indication}: {count}")
    
    logger.info("\n=== ANALYSIS: Herbs by Use Category ===\n")
    if 'use_category' in df:
        category_counts = df['use_category'].value_counts()
        for category, count in category_counts.items():
            print(f"  {category}: {count} ({count/len(df)*100:.1f}%)")

analyse_herb_properties(final_herb_df)

2026-08-21 17:20:18,101 - INFO - 
=== ANALYSIS: Top 10 Herbs by Sanskrit Synonyms ===

2026-08-21 17:20:18,142 - INFO - 
=== ANALYSIS: Most Common Therapeutic Indications ===

2026-08-21 17:20:18,145 - INFO - 
=== ANALYSIS: Herbs by Use Category ===



       name                      sanskrit_synonyms_as_text  synonym_count
      Tulsi        Surasa, Apetarakshasi, Bhuteshta, Gauri              4
       Amla             Amalaki, Dhatri, Vayastha, Vrishya              4
Ashwagandha     Turagi, Vajigandha, Balada, Kushtagandhini              4
      Giloy        Guduchi, Amrita, Madhuparni, Chinnaruha              4
  Shatavari         Shatamuli, Atirasa, Narayani, Bahusuta              4
 Nagarmotha                      Musta, Abda, Jalada, Ghan              4
 Agnimantha         Tarkari, Nadeyi, Vaijayanti, Shriparni              4
  Bhringraj         Keshraja, Markava, Bhringaja, Tekaraja              4
  Jatamansi         Tapaswini, Vilomasa, Bhutajata, Naladi              4
   Chandana Shrikhanda, Gandhasara, Malayaja, Bhadrashriya              4
Top 10 most common indications:
  Raktapitta: 107
  Kushta: 88
  Kasa: 62
  Jwara: 58
  Vrana: 58
  Krimi: 55
  Shwasa: 54
  Mutrakrichha: 53
  Trishna: 52
  Prameha: 46
  Other: 246 (68